# 🎬 TikTok Money Printer — test en ligne (Colab)

Lance l'app sur une machine gratuite de Google et obtiens une **URL publique temporaire** pour la tester depuis n'importe quel navigateur (même ton téléphone).

**Mode d'emploi :** exécute les 3 cellules dans l'ordre (▶️ à gauche de chaque cellule). À la fin, clique sur l'URL `https://....trycloudflare.com` affichée.

💡 **Astuce vitesse** : menu *Exécution → Modifier le type d'exécution → GPU T4* avant de commencer — la transcription Whisper sera bien plus rapide.

⚠️ L'URL et les fichiers disparaissent quand tu fermes le notebook (session Colab temporaire). Télécharge tes clips au fur et à mesure.

## 1️⃣ Installation (2-3 minutes)

In [ ]:
!git clone -q -b claude/tiktok-clip-generator-31rslv https://github.com/Kazza2115/TIKTOKMONEYPRINTER.git
%cd TIKTOKMONEYPRINTER
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print('✅ Installation terminée')

## 2️⃣ Configuration — ta clé API Claude

In [ ]:
import os
from getpass import getpass

os.environ['ANTHROPIC_API_KEY'] = getpass('🔑 Clé API Anthropic (sk-ant-...) : ')
os.environ['APP_PASSWORD'] = getpass("🔒 Mot de passe d'accès à la page (l'URL sera publique) : ")
os.environ['WHISPER_MODEL'] = 'small'
os.environ['WHISPER_DEVICE'] = 'auto'  # utilise le GPU automatiquement si activé
print('✅ Configuration enregistrée')

## 3️⃣ Lancement — clique sur l'URL qui s'affiche

In [ ]:
import re
import subprocess
import sys
import threading
import time

# serveur Flask en arrière-plan
app_proc = subprocess.Popen(
    [sys.executable, '-c', "from app import create_app; create_app().run(host='0.0.0.0', port=5000)"],
)
time.sleep(4)
if app_proc.poll() is not None:
    raise RuntimeError('Le serveur ne démarre pas — relance la cellule 1 puis celle-ci.')

# tunnel public Cloudflare (gratuit, sans compte)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
# vide la sortie du tunnel en tâche de fond pour ne pas le bloquer
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

if url:
    print('\n' + '=' * 60)
    print(f'🌍 TON APP EST EN LIGNE : {url}')
    print('=' * 60)
    print("\nIdentifiant : n'importe quoi — Mot de passe : celui de l'étape 2")
    print('Laisse cette cellule tourner tant que tu utilises la page.')
else:
    print('❌ Tunnel non établi — relance cette cellule.')

---
**Si YouTube bloque le téléchargement** (« Sign in to confirm you're not a bot ») : les IP de Google Cloud sont parfois bloquées. Exporte tes cookies YouTube avec l'extension navigateur *Get cookies.txt LOCALLY*, puis exécute avant de relancer la cellule 3 :
```python
os.environ['YTDLP_COOKIES'] = open('cookies.txt').read()  # après upload du fichier dans Colab
```